# 05 Create SFINCS Scenarios

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import xarray as xr

location_root = Path("..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

from wflow_runs.notebook import load_runtime

runtime = load_runtime(location_root, wflow_domain_review_required=False)
location_name = runtime.location_name
resolve_location_path = runtime.resolve_location_path


## Package Worklist


In [ ]:
scenario_root = runtime.sfincs_scenarios_root
scenario_root.mkdir(parents=True, exist_ok=True)

catalog_path = resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv")
catalog = pd.read_csv(catalog_path, dtype={"event_id": str})

rainfall_members_path = resolve_location_path("data/sources/aorc_sst/rainfall_members.csv")
if rainfall_members_path.exists() and catalog_path.stat().st_mtime < rainfall_members_path.stat().st_mtime:
    raise RuntimeError(
        f"Scenario catalog is older than current AORC SST rainfall members: {catalog_path}. "
        "Rerun 02_flood/03_build_event_catalog.ipynb after the AORC SST Event Windows repair."
    )

historical = catalog.get("event_origin", pd.Series(index=catalog.index, dtype=object)).astype(str).eq("historical_tail")
historical |= catalog.get("catalog_role", pd.Series(index=catalog.index, dtype=object)).astype(str).eq("historical_reference")
if not historical.any():
    raise RuntimeError(
        "Scenario catalog has no historical-tail reference rows. "
        "Rerun 02_flood/03_build_event_catalog.ipynb so historical_tail_catalog.csv is regenerated and merged."
    )

required_aorc_vars = {"APCP_surface", "TMP_2maboveground", "PRES_surface", "DSWRF_surface"}
missing_sources = []
for row in catalog.to_dict("records"):
    member_file = resolve_location_path(row["rainfall_member_file"])
    event_windows_dir = member_file.parent / "event_windows"
    storm_start = pd.Timestamp(row["rainfall_member_time"]).strftime("%Y%m%dT%H")
    source_nc = event_windows_dir / f"{row['rainfall_member_id']}_{storm_start}.nc"
    if not source_nc.exists():
        missing_sources.append({"event_id": row["event_id"], "source_nc": str(source_nc), "missing": "file"})
        continue
    with xr.open_dataset(source_nc) as ds:
        missing = sorted(required_aorc_vars - set(ds.data_vars))
        empty = sorted(
            variable
            for variable in required_aorc_vars & set(ds.data_vars)
            if not bool(np.isfinite(ds[variable]).any().compute().item())
        )
    if missing or empty:
        missing_sources.append({
            "event_id": row["event_id"],
            "source_nc": str(source_nc),
            "missing": ",".join(missing),
            "no_finite_values": ",".join(empty),
        })
if missing_sources:
    raise RuntimeError(
        "Scenario catalog has AORC event-window files missing variables required by Wflow/SFINCS handoff: "
        + str(missing_sources[:10])
    )

worklist = catalog[["event_id"]].drop_duplicates()
worklist.to_csv(runtime.joint_worklist_path, index=False)
worklist.to_csv(runtime.blocked_path, index=False)

pd.Series({
    "worklist_csv": str(runtime.joint_worklist_path),
    "default_worklist_csv": str(runtime.blocked_path),
    "event_count": len(worklist),
    "historical_tail_count": int(historical.sum()),
    "aorc_event_windows_checked": len(catalog),
})


## Sync Handoff


In [ ]:
print(f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only")
print("See cluster/instructions.txt for submit and retrieve commands.")


## Cluster Launch Plan


In [ ]:
import json

joint_worklist_csv = runtime.joint_worklist_path
cluster_plan_path = runtime.sfincs_scenarios_root / "joint_wflow_sfincs_cluster_plan.json"
cluster_plan = {
    "workflow": "Wflow dynamic handoff -> native SFINCS staging -> SFINCS run",
    "slurm_script": "cluster/run_wflow_sfincs_dsai_inland_coupled.slurm",
    "worklist_csv": str(joint_worklist_csv),
    "FORCE_WFLOW": False,
    "OVERWRITE_METEO": False,
}
cluster_plan_path.write_text(json.dumps(cluster_plan, indent=2), encoding="utf-8")

pd.Series(cluster_plan | {"cluster_plan_json": str(cluster_plan_path)}, name="joint_wflow_sfincs_cluster_plan")


## Sample Driver QA


In [ ]:
import json
import matplotlib.pyplot as plt
import xarray as xr

scenarios_root = getattr(runtime, "sfincs_scenarios_root", None)
if scenarios_root is None:
    scenarios_root = paths["scenarios_root"]
manifest_paths = sorted(Path(scenarios_root).rglob("forcing_manifest.json"))
if not manifest_paths:
    raise FileNotFoundError(f"No forcing_manifest.json files under {scenarios_root}")
manifest_path = pd.Series(manifest_paths).sample(1).iloc[0]
event_dir = manifest_path.parent
m = json.loads(manifest_path.read_text())

def resolve_manifest_path(value):
    if not value:
        return None
    path = Path(value)
    if path.exists():
        return path
    if not path.is_absolute():
        candidate = location_root / path
        return candidate if candidate.exists() else path
    parts = path.parts
    if location_root.name in parts:
        candidate = location_root / Path(*parts[parts.index(location_root.name) + 1:])
        return candidate if candidate.exists() else path
    return path

def dataset_mean_series(path):
    with xr.open_dataset(path) as ds:
        da = ds[next(iter(ds.data_vars))]
        return da.mean([d for d in da.dims if d != "time"]).to_series()

panels = []
if (event_dir / "sfincs.bzs").exists():
    wl = pd.read_csv(event_dir / "sfincs.bzs", sep=r"\s+", header=None).set_index(0)
    wl.index = pd.to_timedelta(wl.index, unit="s")
    panels.append((f"{event_dir.name}: boundary water level", [(wl, None)]))
snap = []
for fn in ["snapwave.bhs", "snapwave.btp", "snapwave.bwd", "snapwave.bds"]:
    if (event_dir / fn).exists():
        s = pd.read_csv(event_dir / fn, sep=r"\s+", header=None).set_index(0).mean(axis=1)
        s.index = pd.to_timedelta(s.index, unit="s")
        snap.append((s, fn.replace("snapwave.", "")))
if snap:
    panels.append(("SnapWave boundary drivers", snap))
discharge_path = resolve_manifest_path(m.get("wflow_discharge_forcing"))
if discharge_path and discharge_path.exists():
    panels.append(("Wflow-SFINCS discharge spatial/source mean", [(dataset_mean_series(discharge_path), None)]))
precip_path = event_dir / m.get("netamprfile", "sfincs_netampr.nc")
if precip_path.exists():
    panels.append(("precipitation spatial mean", [(dataset_mean_series(precip_path), None)]))
sm_path = resolve_manifest_path(m.get("soil_moisture_member_file"))
sm_time = m.get("soil_moisture_member_time") or m.get("event_reference_time") or m.get("rainfall_member_time") or m.get("run_start")
if sm_path and sm_path.exists() and sm_time:
    sm_t = pd.Timestamp(sm_time)
    var = m.get("soil_moisture_summary", {}).get("soil_moisture_variable", "SOILSAT_TOP")
    sm = pd.read_csv(sm_path, usecols=["time", var], parse_dates=["time"])
    sm = sm[sm["time"].between(sm_t - pd.Timedelta(days=7), sm_t + pd.Timedelta(days=1))]
    panels.append((f"soil moisture spatial mean ({var})", [(sm.groupby("time")[var].mean(), "soil moisture")]))

fig, axes = plt.subplots(len(panels), 1, figsize=(10, max(2.2 * len(panels), 4)), constrained_layout=True)
if len(panels) == 1:
    axes = [axes]
for ax, (title, series_items) in zip(axes, panels):
    for series, label in series_items:
        series.plot(ax=ax, label=label, legend=False)
    ax.set_title(title)
    if any(label for _, label in series_items):
        ax.legend(ncol=min(len(series_items), 4))
if sm_path and sm_path.exists() and sm_time and panels:
    axes[-1].axvline(pd.Timestamp(sm_time), color="k", ls="--", lw=1)
event_dir
